In [ ]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import librosa
import torch
from tqdm import tqdm


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
processor = WhisperProcessor.from_pretrained("openai/whisper-large")

In [ ]:

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-large")
model = model.to(device)

In [ ]:
test_audio, sr = librosa.load('meeting.wav', sr=16_000)
inputs = processor(
    test_audio,
    return_tensors="pt",
    truncation=False,
    padding="longest",
    return_attention_mask=True,
    sampling_rate=16_000
)

In [ ]:
generated_ids = model.generate(return_timestamps=True, language="ru", **inputs.to(device))

In [ ]:
decoded = processor.batch_decode(generated_ids.cpu())
transcription = decoded[0]


In [ ]:
with open('transcription.txt', 'w+') as f:
    f.write(transcription[0])

In [ ]:
with open('transcription.txt', 'r') as f:
    transcription = f.read()

del model

In [ ]:
from transformers import GPT2Tokenizer, T5ForConditionalGeneration 

tokenizer = GPT2Tokenizer.from_pretrained('RussianNLP/FRED-T5-Summarizer',eos_token='</s>')


In [ ]:
model = T5ForConditionalGeneration.from_pretrained('RussianNLP/FRED-T5-Summarizer')
model = model.to(device)

In [ ]:
def chunk_text(text, max_tokens=1024):
    sentences = text.split('. ')
    chunks, chunk = [], ''
    current_len = 0

    for sentence in sentences:
        token_len = len(tokenizer.encode(sentence, add_special_tokens=False))
        if current_len + token_len > max_tokens:
            chunks.append(chunk.strip())
            chunk = sentence + '. '
            current_len = token_len
        else:
            chunk += sentence + '. '
            current_len += token_len
    if chunk:
        chunks.append(chunk.strip())
    return chunks

def tokenize_chunk(chunk):
    input_text = "<LM> Сократи текст.\n " + chunk
    input_ids = torch.tensor([tokenizer.encode(input_text)]).to(device)
    return input_ids

def summarize_chunk(input_ids):
    summary_ids = model.generate(
        input_ids,
        eos_token_id=tokenizer.eos_token_id,
        num_beams=5,
        min_new_tokens=17,
        max_new_tokens=200,
        do_sample=True,
        no_repeat_ngram_size=4,
        top_p=0.9
    )
    return tokenizer.decode(summary_ids[0][1:])

In [ ]:
chunks = chunk_text(transcription)
tokenized_chunks = [tokenize_chunk(c) for c in chunks]

In [ ]:
summaries = [summarize_chunk(c) for c in tokenized_chunks]

with open('summary.txt', 'w+') as f:
    f.write('\n'.join(summaries))